In [1]:
# 실습에 필요한 패키지를 설치합니다.
!pip install -r ../requirements.txt

# 05 Agentic

`agentic`이라는 말은 LLM이 사람처럼 완전히 알아서 행동한다는 뜻이 아니다.
LLM 애플리케이션이 목표를 기준으로 계획을 세우고, 도구를 사용하고, 결과를 점검하고, 필요하면 다시 시도하는 구조를 가진다는 뜻에 가깝다.

즉 agentic workflow의 핵심은 한 번의 답변 생성이 아니라 `상태를 가진 반복 실행`이다.
사용자의 목표가 있고, 현재 state가 있고, 다음 action을 고르고, action 결과를 다시 state에 반영한 뒤, 충분할 때까지 이어간다.

## 이번 노트에서 볼 것

- agentic이란 무엇인가
- planning
- reflection
- iterative tool use
- human approval
- evaluation

## 1. agentic이란 무엇인가

Agentic workflow는 LLM이 단순히 답변만 만드는 구조가 아니라, 목표를 달성하기 위해 여러 단계를 스스로 이어가는 구조다.

조금 더 구체적으로 말하면 아래 요소가 들어간다.

- goal: 무엇을 달성해야 하는가
- state: 지금까지 무엇을 알고 있고 무엇을 했는가
- plan: 어떤 순서로 풀 것인가
- action: 다음에 어떤 도구나 작업을 실행할 것인가
- observation: action의 결과로 무엇을 얻었는가
- reflection: 결과가 충분한지, 문제가 없는지 점검했는가
- evaluation: 최종 결과가 기준을 만족하는가

그래서 agentic을 이해할 때는 `똑똑한 답변`보다 `반복 가능한 문제 해결 루프`로 보는 편이 더 좋다.

## 2. Planning

`planning`은 사용자의 요청을 바로 실행하지 않고, 먼저 해결 순서를 만드는 단계다.

예를 들어 사용자가 "우리 서비스 환불 정책을 보고 고객에게 답변해줘"라고 했다면, agent는 바로 답변을 쓰기 전에 이런 계획을 세울 수 있다.

1. 질문의 의도를 정리한다
2. 환불 정책 문서를 찾는다
3. 관련 조항을 추린다
4. 고객에게 보낼 답변 초안을 만든다
5. 답변이 정책과 충돌하지 않는지 점검한다

여기서 중요한 점은 plan이 정답이 아니라는 것이다.  
계획은 현재 정보로 세운 가설이고, tool 사용 결과나 reflection 결과에 따라 바뀔 수 있다.

In [2]:
from pprint import pprint
from typing import TypedDict


class AgentState(TypedDict, total=False):
    goal: str
    plan: list[str]
    step_index: int
    tool_results: list[str]
    reflections: list[str]
    approval_required: bool
    approved: bool
    final_answer: str
    evaluation: dict[str, object]


def planning(state: AgentState) -> AgentState:
    goal = state["goal"]
    return {
        "plan": [
            f"사용자 목표를 정리한다: {goal}",
            "필요한 근거를 tool로 확인한다",
            "확인한 근거를 바탕으로 답변 초안을 만든다",
            "답변의 누락과 위험을 reflection으로 점검한다",
            "최종 답변을 evaluation 기준으로 검사한다",
        ],
        "step_index": 0,
        "tool_results": [],
        "reflections": [],
        "approval_required": False,
        "approved": False,
    }


state = {"goal": "환불 정책을 근거로 고객 답변을 작성한다"}
state.update(planning(state))
pprint(state)

{'approval_required': False,
 'approved': False,
 'goal': '환불 정책을 근거로 고객 답변을 작성한다',
 'plan': ['사용자 목표를 정리한다: 환불 정책을 근거로 고객 답변을 작성한다',
          '필요한 근거를 tool로 확인한다',
          '확인한 근거를 바탕으로 답변 초안을 만든다',
          '답변의 누락과 위험을 reflection으로 점검한다',
          '최종 답변을 evaluation 기준으로 검사한다'],
 'reflections': [],
 'step_index': 0,
 'tool_results': []}


위 예시에서 plan은 별도의 문서가 아니라 state 안에 들어간다.
이렇게 하면 뒤 단계가 같은 state를 보면서 지금 몇 번째 단계인지, 어떤 근거를 모았는지, 무엇을 다시 해야 하는지 판단할 수 있다.

## 3. Reflection

`reflection`은 agent가 자신의 중간 결과를 다시 점검하는 단계다.

LLM은 그럴듯한 답을 빨리 만들 수 있지만, 그 답이 충분한지까지 항상 보장하지는 않는다.
그래서 agentic workflow에서는 중간중간 아래 질문을 던진다.

- 지금 답변에 근거가 충분한가?
- 사용자의 원래 목표를 놓친 부분은 없는가?
- tool 결과와 답변이 서로 충돌하지 않는가?
- 위험하거나 사람 확인이 필요한 action은 없는가?

reflection은 단순한 자기비판이 아니라 다음 action을 고르기 위한 상태 점검이다.

In [3]:
def reflection(state: AgentState) -> AgentState:
    tool_results = state.get("tool_results", [])

    if not tool_results:
        message = "아직 근거가 없다. 먼저 tool을 사용해야 한다."
        return {"reflections": state.get("reflections", []) + [message]}

    latest = tool_results[-1]
    if "찾지 못함" in latest:
        message = "관련 근거를 찾지 못했다. 검색 전략을 바꿔야 한다."
    elif "상품 수령 후 7일 이내" in latest:
        message = "환불 가능 조건을 찾았다. 고객 답변 초안을 만들 수 있다."
    else:
        message = "근거는 있지만 최종 답변에 쓰기 전에 조건을 더 확인해야 한다."

    return {"reflections": state.get("reflections", []) + [message]}


state.update(reflection(state))
pprint(state["reflections"])

['아직 근거가 없다. 먼저 tool을 사용해야 한다.']


## 4. Iterative Tool Use

`iterative tool use`는 tool을 한 번만 호출하는 것이 아니라, 결과를 보고 다음 tool 사용을 이어가는 방식이다.

일반적인 tool call은 아래처럼 보인다.

`사용자 질문 -> tool 호출 -> 답변`

agentic tool use는 조금 다르다.

`목표 확인 -> tool 호출 -> 결과 관찰 -> reflection -> 다음 tool 결정 -> 다시 실행`

즉 tool은 답변을 만들기 위한 한 번짜리 부품이 아니라, 문제 해결 루프 안에서 반복적으로 사용되는 action이다.

In [4]:
POLICY_DOCS = {
    "refund_policy": "상품 수령 후 7일 이내이고 사용 흔적이 없으면 환불 가능하다.",
    "business_hours": "고객센터 운영 시간은 평일 10시부터 18시까지다.",
}


def search_policy(query: str) -> str:
    if "환불" in query:
        return POLICY_DOCS["refund_policy"]
    return "찾지 못함"


def draft_answer(evidence: str) -> str:
    if "상품 수령 후 7일 이내" in evidence:
        return f"정책 확인 결과, {evidence} 따라서 조건을 만족하면 환불 안내가 가능합니다."
    return "현재 근거만으로는 확정 답변을 만들기 어렵습니다."


def run_tool_step(state: AgentState) -> AgentState:
    step_index = state.get("step_index", 0)
    plan = state["plan"]
    current_step = plan[step_index]

    if "tool" in current_step:
        result = search_policy(state["goal"])
    elif "초안" in current_step:
        evidence = state.get("tool_results", ["찾지 못함"])[-1]
        result = draft_answer(evidence)
    else:
        result = "실행 완료"

    return {
        "tool_results": state.get("tool_results", []) + [result],
        "step_index": min(step_index + 1, len(plan) - 1),
    }


state.update(run_tool_step(state))
state.update(reflection(state))
pprint(state)

{'approval_required': False,
 'approved': False,
 'goal': '환불 정책을 근거로 고객 답변을 작성한다',
 'plan': ['사용자 목표를 정리한다: 환불 정책을 근거로 고객 답변을 작성한다',
          '필요한 근거를 tool로 확인한다',
          '확인한 근거를 바탕으로 답변 초안을 만든다',
          '답변의 누락과 위험을 reflection으로 점검한다',
          '최종 답변을 evaluation 기준으로 검사한다'],
 'reflections': ['아직 근거가 없다. 먼저 tool을 사용해야 한다.',
                 '근거는 있지만 최종 답변에 쓰기 전에 조건을 더 확인해야 한다.'],
 'step_index': 1,
 'tool_results': ['실행 완료']}


## 5. Human Approval

`human approval`은 agent가 모든 action을 바로 실행하지 않고, 특정 조건에서는 사람의 승인을 기다리게 만드는 장치다.

특히 아래 경우에는 approval gate가 필요하다.

- 메일 발송, 결제, 환불 처리처럼 외부 세계에 영향을 주는 action
- 파일 삭제, 배포, 데이터 변경처럼 되돌리기 어려운 action
- 법무, 의료, 재무처럼 오답의 비용이 큰 판단
- agent가 근거 부족이나 정책 충돌을 감지한 경우

중요한 점은 사람 승인이 agentic의 반대가 아니라는 것이다.
좋은 agentic workflow는 스스로 할 수 있는 일과 사람에게 넘겨야 하는 일을 구분한다.

In [5]:
def approval_gate(state: AgentState) -> AgentState:
    latest_result = state.get("tool_results", [""])[-1]
    needs_approval = "상품 수령 후 7일 이내" in latest_result or "환불 안내" in latest_result

    if not needs_approval:
        return {"approval_required": False, "approved": True}

    return {
        "approval_required": True,
        "approved": False,
        "reflections": state.get("reflections", [])
        + ["고객에게 환불 안내를 보내기 전 사람 승인이 필요하다."],
    }


state.update(approval_gate(state))
pprint({"approval_required": state["approval_required"], "approved": state["approved"]})

{'approval_required': False, 'approved': True}


실제 서비스에서는 이 지점에서 실행을 잠시 멈추고, 사람에게 아래 정보를 보여준다.

- agent의 목표
- 실행하려는 action
- 사용한 근거
- 예상 결과
- 승인, 수정, 거절 버튼

승인되면 다음 단계로 가고, 수정되면 state를 바꿔 다시 실행하고, 거절되면 workflow를 종료하거나 fallback 경로로 보낸다.

## 6. Evaluation

`evaluation`은 최종 결과가 기준을 만족하는지 검사하는 단계다.

reflection이 중간 점검이라면, evaluation은 최종 점검에 가깝다.
agentic workflow에서는 결과가 만들어졌다는 사실만으로 끝내지 않고, 기준을 통과했는지 확인한다.

예를 들어 고객 답변 agent라면 아래 기준을 둘 수 있다.

- 정책 근거가 포함되어 있는가
- 사용자의 질문에 직접 답했는가
- 모르는 내용을 단정하지 않았는가
- 사람 승인이 필요한 경우 승인 없이 외부 action을 하지 않았는가

evaluation은 LLM에게 맡길 수도 있고, 규칙 기반 검사나 테스트 코드로 만들 수도 있다.

In [6]:
def evaluate(state: AgentState) -> AgentState:
    final_answer = state.get("final_answer", "")
    has_policy_evidence = "정책" in final_answer or "7일 이내" in final_answer
    respects_approval = not state.get("approval_required") or state.get("approved")
    answers_goal = "환불" in final_answer

    checks = {
        "has_policy_evidence": has_policy_evidence,
        "respects_approval": respects_approval,
        "answers_goal": answers_goal,
    }
    score = sum(checks.values()) / len(checks)

    return {
        "evaluation": {
            "checks": checks,
            "score": score,
            "passed": score == 1.0,
        }
    }


state["final_answer"] = "정책 확인 결과, 상품 수령 후 7일 이내이고 사용 흔적이 없으면 환불 가능합니다."
state["approved"] = True
state.update(evaluate(state))
pprint(state["evaluation"])

{'checks': {'answers_goal': True,
            'has_policy_evidence': True,
            'respects_approval': True},
 'passed': True,
 'score': 1.0}


## 7. 하나로 묶어보기

이제 planning, tool use, reflection, approval, evaluation을 하나의 작은 workflow로 묶어보자.

아래 코드는 실제 LLM agent는 아니지만 agentic 구조를 이해하기 위한 최소 예시다.
핵심은 `while` 루프 안에서 state가 계속 갱신되고, 그 state를 기준으로 다음 행동이 결정된다는 점이다.

In [ ]:
def run_agentic_workflow(goal: str) -> AgentState:
    state: AgentState = {"goal": goal}
    state.update(planning(state))

    while state["step_index"] < len(state["plan"]):
        state.update(run_tool_step(state))
        state.update(reflection(state))
        state.update(approval_gate(state))

        if state.get("approval_required") and not state.get("approved"):
            state["reflections"] = state.get("reflections", []) + [
                "실제 서비스에서는 여기서 workflow를 멈추고 사람 승인을 기다린다."
            ]
            break

        if state["step_index"] == len(state["plan"]) - 1:
            break

    evidence = state.get("tool_results", ["근거 없음"])[-1]
    state["final_answer"] = draft_answer(evidence)

    if state.get("approval_required"):
        state["approved"] = True
        state["reflections"] = state.get("reflections", []) + [
            "실습에서는 사람이 승인했다고 가정하고 evaluation까지 진행한다."
        ]

    state.update(evaluate(state))
    return state


result = run_agentic_workflow("고객에게 환불 가능 여부를 안내한다")
pprint(result)

{'approval_required': True,
 'approved': True,
 'evaluation': {'checks': {'answers_goal': True,
                           'has_policy_evidence': True,
                           'respects_approval': True},
                'passed': True,
                'score': 1.0},
 'final_answer': '정책 확인 결과, 상품 수령 후 7일 이내이고 사용 흔적이 없으면 환불 가능하다. 따라서 조건을 만족하면 '
                 '환불 안내가 가능합니다.',
 'goal': '고객에게 환불 가능 여부를 안내한다',
 'plan': ['사용자 목표를 정리한다: 고객에게 환불 가능 여부를 안내한다',
          '필요한 근거를 tool로 확인한다',
          '확인한 근거를 바탕으로 답변 초안을 만든다',
          '답변의 누락과 위험을 reflection으로 점검한다',
          '최종 답변을 evaluation 기준으로 검사한다'],
 'reflections': ['근거는 있지만 최종 답변에 쓰기 전에 조건을 더 확인해야 한다.',
                 '환불 가능 조건을 찾았다. 고객 답변 초안을 만들 수 있다.',
                 '고객에게 환불 안내를 보내기 전 사람 승인이 필요하다.',
                 '실제 서비스에서는 여기서 workflow를 멈추고 사람 승인을 기다린다.',
                 '실습에서는 사람이 승인했다고 가정하고 evaluation까지 진행한다.'],
 'step_index': 2,
 'tool_results': ['실행 완료', '상품 수령 후 7일 이내이고 사용 흔적이 없으면 환불 가능하다.']}


위 예시는 단순하지만 agentic workflow의 뼈대를 모두 가지고 있다.

- planning: 먼저 실행 순서를 만든다
- iterative tool use: state를 보며 tool을 반복 실행한다
- reflection: tool 결과가 충분한지 점검한다
- human approval: 위험하거나 외부 영향을 주는 action 앞에서 멈춘다
- evaluation: 최종 결과가 기준을 통과하는지 확인한다

실제 agent는 여기에 LLM 호출, LangGraph node, 외부 API tool, memory, retry, fallback 등이 더해진 형태라고 보면 된다.

## 정리

이번 노트에서 기억하면 좋은 문장은 아래 다섯 개다.

- agentic은 한 번의 답변이 아니라 목표 기반 반복 workflow다
- planning은 실행 순서를 만드는 단계지만, 결과에 따라 바뀔 수 있다
- reflection은 중간 결과를 보고 다음 action을 조정하는 상태 점검이다
- tool use는 observation과 함께 반복될 때 agentic해진다
- human approval과 evaluation은 agent를 더 느리게 만드는 장치가 아니라 더 안전하게 만드는 장치다

다음 단계에서는 이 구조를 LangGraph로 옮겨서 각 요소를 node로 나누고, state와 edge로 agentic loop를 표현해보면 된다.